In [1]:
from dotenv import load_dotenv
from langchain.prompts import ChatPromptTemplate
from langchain.schema.output_parser import StrOutputParser
from langchain.schema.runnable import RunnableParallel, RunnableLambda
from langchain_openai import ChatOpenAI

In [2]:
load_dotenv()

True

In [3]:
model = ChatOpenAI(
    model="gpt-4o",
    temperature=0,
    max_tokens=150 
)

In [7]:
prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an expert product reviewer."),
        ("human", "List the main features of the product {product_name}."),
    ]
)


In [8]:
# Define pros analysis step
def analyze_pros(features):
    pros_template = ChatPromptTemplate.from_messages(
        [
            ("system", "You are an expert product reviewer."),
            (
                "human",
                "Given these features: {features}, list the pros of these features.",
            ),
        ]
    )
    return pros_template.format_prompt(features=features)

In [9]:
# Define cons analysis step
def analyze_cons(features):
    cons_template = ChatPromptTemplate.from_messages(
        [
            ("system", "You are an expert product reviewer."),
            (
                "human",
                "Given these features: {features}, list the cons of these features.",
            ),
        ]
    )
    return cons_template.format_prompt(features=features)

In [10]:
# Combine pros and cons into a final review
def combine_pros_cons(pros, cons):
    return f"Pros:\n{pros}\n\nCons:\n{cons}"

In [11]:
# Simplify branches with LCEL
pros_branch_chain = (
    RunnableLambda(lambda x: analyze_pros(x)) | model | StrOutputParser()
)

cons_branch_chain = (
    RunnableLambda(lambda x: analyze_cons(x)) | model | StrOutputParser()
)

In [12]:
chain = (
    prompt_template
    | model
    | StrOutputParser()
    | RunnableParallel(branches={"pros": pros_branch_chain, "cons": cons_branch_chain})
    | RunnableLambda(lambda x: combine_pros_cons(x["branches"]["pros"], x["branches"]["cons"]))
)

In [13]:
result = chain.invoke({"product_name": "MacBook Pro"})
print(result)

Pros:
The MacBook Pro's features offer several advantages that make it a compelling choice for users seeking a high-performance laptop. Here are the pros of the highlighted features:

1. **Apple Silicon Chips**:
   - **Performance**: The M1, M1 Pro, M1 Max, M2, and M2 Pro/Max chips deliver exceptional performance, making the MacBook Pro capable of handling demanding tasks such as video editing, 3D rendering, and software development with ease.
   - **Energy Efficiency**: These chips are designed to be highly energy-efficient, resulting in longer battery life, which is ideal for users who need to work on the go without frequent recharging.
   - **Unified Memory Architecture**: The integration of CPU

Cons:
While the latest MacBook Pro models boast impressive features, there are some potential downsides to consider:

1. **Apple Silicon Chips**:
   - **Compatibility Issues**: Some older software and applications may not be fully optimized for Apple Silicon, potentially leading to compatib